In [ ]:
pip install pandas numpy scikit-learn xgboost shap imbalanced-learn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

def load_data(path):
    df = pd.read_csv('UNSW-NB15.CSV')

    # Convert label to binary
    if 'label' in df.columns:
        df['label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

    # Encode categorical
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    X = df.drop('label', axis=1)
    y = df['label']

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = XGBClassifier(n_estimators=100, max_depth=5)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

    joblib.dump(model, "models/model.pkl")

    return model, X_test

In [ ]:
import shap

def get_shap_values(model, X_sample):
    explainer = shap.Explainer(model)
    shap_values = explainer(X_sample)
    return shap_values

In [ ]:
def get_top_k(shap_values, k=10):
    import numpy as np
    vals = np.abs(shap_values.values).mean(axis=0)
    top_k = np.argsort(vals)[-k:]
    return set(top_k)

In [ ]:
def add_noise(X, epsilon=0.01):
    noise = np.random.normal(0, 1, X.shape)
    return X + epsilon * noise

In [ ]:
def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

In [ ]:
def baseline_test(model, X_test):
    X_sample = X_test[:200]

    # Original
    shap_orig = get_shap_values(model, X_sample)
    top_orig = get_top_k(shap_orig)

    # Noisy
    X_noisy = add_noise(X_sample, epsilon=0.05)
    shap_noisy = get_shap_values(model, X_noisy)
    top_noisy = get_top_k(shap_noisy)

    score = jaccard_similarity(top_orig, top_noisy)

    print("Baseline Stability:", score)

    return score

In [ ]:
def stability_aware_training(X, y):
    # Add noisy data
    X_noisy = add_noise(X, epsilon=0.05)

    # Combine original + noisy
    X_aug = np.vstack((X, X_noisy))
    y_aug = np.hstack((y, y))

    return train_model(X_aug, y_aug)

In [ ]:
def compare_models(X, y):
    print("=== Baseline Model ===")
    model_base, X_test = train_model(X, y)
    base_score = baseline_test(model_base, X_test)

    print("\n=== Stability-Aware Model ===")
    model_new, X_test2 = stability_aware_training(X, y)
    new_score = baseline_test(model_new, X_test2)

    print("\nImprovement:", new_score - base_score)

In [ ]:
if __name__ == "__main__":
    X, y = load_data("data/unsw.csv")

    compare_models(X, y)

In [ ]:
X, y = load_data("data/iot23.csv")

In [ ]:
for eps in [0.01, 0.05, 0.1]:
    print(f"\nNoise Level: {eps}")
    X_noisy = add_noise(X_test, eps)

In [ ]:
X_sample = X_test[:100]